In [ ]:
from datetime import date, timedelta, datetime
from gdeltdoc import GdeltDoc, Filters
from urllib.parse import quote
import yfinance as yf
import pandas as pd
import time
import os

START_DATE = date(year=2018, month=1, day=1)
END_DATE = datetime.now().date()
DATE_INCREMENT = timedelta(days=31)

gdd = GdeltDoc()

In [ ]:
# Get S&P 500 company names
sp500 = pd.read_csv("sp500.csv")
tickers = sorted([ticker for ticker in sp500["Symbol"]])
symbol_to_security = sp500.set_index("Symbol")["Security"].to_dict()

In [ ]:
# Save S&P 500 company data
sp500_data = yf.download(tickers, start="2018-01-01", end="2025-09-01", group_by="ticker")
sp500_data.to_csv("sp500_data.csv")

In [ ]:
def save_articles_to_file(ticker: str, start_date: str, end_date: str, *, save_file: str):
    filters = Filters(keyword=quote(symbol_to_security[ticker]), start_date=start_date, end_date=end_date)
    articles = gdd.article_search(filters)

    if len(articles) > 0:
        articles = articles[articles["language"] == "English"]
        articles = articles[articles["sourcecountry"] == "United States"]

        os.makedirs(os.path.dirname(save_file), exist_ok=True)
        articles.to_csv(save_file)

In [ ]:
bad_data_count = 0

start = time.time()

selected_tickers = tickers[0:101]

try:
    for ticker in selected_tickers:
        if len(symbol_to_security[ticker]) < 5:
            bad_data_count += 1
            print(f"{'\033[38;5;1m'}BAD DATA{'\033[0m'}: {'\033[38;5;3m'}{ticker}{'\033[0m'}")
            continue

        start_date = START_DATE
        while start_date + DATE_INCREMENT < END_DATE:
            save_file = f"cached_articles/{start_date}/{ticker}.csv"

            if os.path.exists(save_file):
                print(f"{'\033[38;5;5m'}SKIPPING DATA{'\033[0m'}: {'\033[38;5;7m'}{start_date} {'\033[38;5;3m'}{ticker}{'\033[0m'}")
            else:
                print(f"{'\033[38;5;6m'}DOWNLOADING DATA{'\033[0m'}: {'\033[38;5;7m'}{start_date} {'\033[38;5;3m'}{ticker}{'\033[0m'}")
                save_articles_to_file(ticker, str(start_date), str(start_date + DATE_INCREMENT), save_file=save_file)

            start_date += DATE_INCREMENT
        break
except Exception as err:
    print(f"{'\033[38;5;1m'}FAILED TO GET ALL DATA{'\033[0m'}")
    raise err

print(f"TOTAL TIME: {time.time() - start:.2f} (s)")
print(f"TOTAL BAD DATA: {bad_data_count}")

In [ ]:
print(tickers[0:101])  # indices 0–100 (DYLAN)
print(tickers[101:202])  # indices 101–201 (EFORD)
print(tickers[202:303])  # indices 202–302 (JACKY)
print(tickers[303:403])  # indices 303–402 (CALVIN)
print(tickers[403:503])  # indices 403–502 (TEJU)
